# SOM-VAE для классификации спутниковых изображений EuroSAT

## Импорты и загрузка данных

In [ ]:
import optuna

# Основные библиотеки
import numpy as np
import matplotlib.pyplot as plt

# PyTorch
import torch
from torch import nn
from torch.functional import F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# Обработка данных
import polars as pl
import os
from PIL import Image
from pathlib import Path

import random
# Метрики
from sklearn.metrics import normalized_mutual_info_score

# Прогресс-бары
from tqdm.notebook import tqdm

# Загрузка датасетов
import kagglehub

## Загрузка датасета EuroSAT

In [ ]:
# Download latest version
path = kagglehub.dataset_download("apollo2506/eurosat-dataset")

print("Path to dataset files:", path)
img_path = os.path.join(path, "EuroSAT")

## Создание Dataset класса

In [ ]:
class EuroSATDataset(Dataset):
    def __init__(self, csv_path, image_root, transform=None, filt_class=None, return_label=False):
        self.df = pl.read_csv(csv_path).drop(pl.read_csv(csv_path).columns[0])
        self.image_root = Path(image_root)
        self.transform = transform  # ← внешний трансформ
        self.return_label = return_label

        if filt_class is not None:
            self.df = self.df.filter(pl.col("Label") == filt_class)
            
    def __len__(self):
        return self.df.height

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)

        img_path = self.image_root / row["Filename"]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)  # ← применяем внешний трансформ

        if self.return_label:
            return image, row["Label"], row["ClassName"]

        return image

## Создание датасетов

In [ ]:
train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomChoice([
        T.Lambda(lambda x: x),                          # 0°
        T.Lambda(lambda x: x.rotate(90, expand=False)),   # 90°
        T.Lambda(lambda x: x.rotate(180, expand=False)),  # 180°
        T.Lambda(lambda x: x.rotate(270, expand=False)),  # 270°
    ]),
    T.ColorJitter(brightness=0.05, contrast=0.05),
    T.RandomApply([T.GaussianBlur(3, (0.1, 0.3))], p=0.2),
    T.Resize((64, 64)),  # Для надёжности (изображения EuroSAT уже 64×64)
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # → [-1, 1]
])

val_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
# Создание датасетов
train_dataset = EuroSATDataset(
    csv_path=os.path.join(img_path, "train.csv"),
    image_root=img_path,
    transform=train_transform,
    return_label=True
)

val_dataset = EuroSATDataset(
    csv_path=os.path.join(img_path, "validation.csv"),
    image_root=img_path,
    transform=val_transform,
    return_label=True
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

## Демонстрация изображений

In [ ]:
# Показываем случайное изображение из тренировочного датасета
idx = np.random.randint(len(train_dataset))
x, label, name = train_dataset[idx]

print(f"Image shape: {x.shape}")
print(f"Label: {label} ({name})")

# Денормализация для отображения
img_display = x.permute(1, 2, 0) * 0.5 + 0.5
plt.figure(figsize=(6, 6))
plt.imshow(img_display)
plt.title(f"Class: {name}")
plt.axis('off')
plt.show()

## Просмотр примеров по классам

In [ ]:
def plot_n_rand_class(dataset, class_name, n=8):
    """Показывает n случайных изображений указанного класса"""
    idxs = dataset.df \
        .with_columns(
            pl.arange(0, pl.len()).alias("idx")
        ) \
        .filter(
            pl.col("ClassName") == class_name
        ) \
        .select("idx").to_numpy().ravel()
    
    n_rand_idxs = np.random.choice(idxs, size=min(n, len(idxs)), replace=False)
    
    fig = plt.figure(figsize=(n, 2))
    fig.suptitle(f"Examples of {class_name}")
    for i, idx in enumerate(n_rand_idxs):
        x, label, class_name = dataset[idx]
        ax = fig.add_subplot(1, len(n_rand_idxs), i + 1)
        ax.imshow(x.permute(1, 2, 0) * 0.5 + 0.5)
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()

# Показываем примеры для каждого класса
unique_classes = train_dataset.df["ClassName"].unique()
print(f"Classes in dataset: {unique_classes}")

for class_name in unique_classes:
    plot_n_rand_class(train_dataset, class_name, n=6)

## Статистика датасета

In [ ]:
# Анализ распределения классов
class_counts = train_dataset.df["ClassName"].value_counts().sort("ClassName")
print("Class distribution:")
print(class_counts)

# Визуализация распределения
plt.figure(figsize=(12, 6))
plt.bar(class_counts["ClassName"], class_counts["count"])
plt.title("Class Distribution in EuroSAT Dataset")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
class GlobalSpatialSOMLayer(nn.Module):
    def __init__(self, grid_h, grid_w, channels, latent_h, latent_w, alpha_grad=1.0):
        super().__init__()
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.c, self.h, self.w = channels, latent_h, latent_w
        self.template_dim = channels * latent_h * latent_w
        self.embeddings = nn.Parameter(torch.randn(grid_h * grid_w, self.template_dim))
        self.alpha_grad = alpha_grad  # вес градиентного члена

    def _compute_distances(self, z_e_flat, embeddings_flat):
        """
        z_e_flat: [B, D]
        embeddings_flat: [K, D]
        Возвращает: [B, K] — gradient-aware расстояния
        """
        B, D = z_e_flat.shape
        K = embeddings_flat.shape[0]
        C, H, W = self.c, self.h, self.w
    
        # Восстанавливаем форму изображений
        z_e = z_e_flat.view(B, C, H, W)          # [B, C, H, W]
        emb = embeddings_flat.view(K, C, H, W)    # [K, C, H, W]
    
        # Расширяем для попарного сравнения
        z_e_exp = z_e.unsqueeze(1)                # [B, 1, C, H, W]
        emb_exp = emb.unsqueeze(0)                # [1, K, C, H, W]
    
        diff = z_e_exp - emb_exp                  # [B, K, C, H, W]
    
        # 1. Пиксельный L2
        l2_pixel = diff.pow(2).mean(dim=[2, 3, 4])  # [B, K]
    
        if self.h <= 1 or self.w <= 1:
            return l2_pixel
    
        # 2. Горизонтальные градиенты (по высоте)
        grad_h = diff[:, :, :, 1:, :] - diff[:, :, :, :-1, :]  # [B, K, C, H-1, W]
        # 3. Вертикальные градиенты (по ширине)
        grad_w = diff[:, :, :, :, 1:] - diff[:, :, :, :, :-1]  # [B, K, C, H, W-1]
    
        l2_grad_h = grad_h.pow(2).mean(dim=[2, 3, 4]) if grad_h.shape[3] > 0 else 0
        l2_grad_w = grad_w.pow(2).mean(dim=[2, 3, 4]) if grad_w.shape[4] > 0 else 0
    
        l2_grad = l2_grad_h + l2_grad_w
    
        return l2_pixel + self.alpha_grad * l2_grad

    def forward(self, z_e):
        batch_size = z_e.size(0)
        z_e_flat = z_e.view(batch_size, -1)  # [B, D]
    
        # Считаем расстояния с учётом градиентов
        distances = self._compute_distances(z_e_flat, self.embeddings)  # [B, K]
    
        indices = torch.argmin(distances, dim=1)  # [B]
    
        z_q_flat = self.embeddings[indices]  # [B, D]
        z_q = z_q_flat.view(batch_size, self.c, self.h, self.w)
    
        return z_q, indices

    def get_neighbors(self, indices):
        # Логика поиска соседей по 2D сетке
        row = indices // self.grid_w
        col = indices % self.grid_w
        
        def get_idx(r, c):
            r = torch.clamp(r, 0, self.grid_h - 1)
            c = torch.clamp(c, 0, self.grid_w - 1)
            return r * self.grid_w + c

        return torch.stack([
            get_idx(row-1, col), get_idx(row+1, col), 
            get_idx(row, col-1), get_idx(row, col+1)
        ], dim=1)

    def calc_activations(self, z_e):
        batch_size = z_e.size(0)
        z_e_flat = z_e.view(batch_size, -1)
        
        distances = self._compute_distances(z_e_flat, self.embeddings)  # [B, K]
        
        # Инвертируем, чтобы близость → большое число
        activations = 1.0 / (distances + 1e-8)
        
        return activations

In [ ]:
class EuroSAT_GlobalSOM_Deep(nn.Module):
    def __init__(self, in_channels=3, grid_size=(16, 16), latent_dim=(32, 4, 4), num_classes=10):
        super().__init__()
        c_l, h_l, w_l = latent_dim

        # ---------------------
        # 1. ЭНКОДЕР (64x64 -> 8x8)
        # ---------------------
        self.encoder = nn.Sequential(
            # Block 1: 64x64 -> 32x32
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 32, 4, stride=2, padding=1),  # 64 -> 32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            
            # Block 2: 32x32 -> 16x16
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 4, stride=2, padding=1),  # 32 -> 16
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            
            # Block 3: 16x16 -> 8x8
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 4, stride=2, padding=1),  # 16 -> 8
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
        
            # Block 4: 8x8 -> 4x4
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),  # ✅ Исправлено: было 128 → теперь 64
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, latent_dim[0], 4, stride=2, padding=1),  # 8 -> 4 ✅ Исправлен комментарий
            nn.BatchNorm2d(latent_dim[0]),
            nn.LeakyReLU(0.2),
        )

        # ---------------------
        # 2. SOM СЛОЙ
        # ---------------------
        self.som = GlobalSpatialSOMLayer(grid_size[0], grid_size[1], c_l, h_l, w_l)

        # ---------------------
        # 3. ДЕКОДЕР (8x8 -> 64x64)
        # ---------------------
        self.decoder = nn.Sequential(
            # 4 -> 8
            nn.ConvTranspose2d(latent_dim[0], 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            
            # 8 -> 16
            nn.ConvTranspose2d(64, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
        
            # 16 -> 32
            nn.ConvTranspose2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
        
            # 32 -> 64
            nn.ConvTranspose2d(128, in_channels, 4, stride=2, padding=1),
            nn.Tanh()
        )


        # ---------------------
        # 4. Классификатор
        # ---------------------
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c_l * h_l * w_l, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, indices = self.som(z_e)
        logits = self.classifier(z_q)
        x_hat_e = self.decoder(z_e)
        x_hat_q = self.decoder(z_q)
        return x_hat_e, x_hat_q, z_e, z_q, indices, logits

    def decode(self, z):
        return self.decoder(z)


## Loss Function

In [ ]:
def som_vae_loss(x, x_hat_e, x_hat_q, z_e, z_q, indices, logits, targets, som_layer, 
                 alpha=1.0, beta=1.0, gamma=1.0):
    """
    Комбинированная функция потерь для SOM-VAE
    
    Args:
        x: исходные изображения
        x_hat_e: реконструкция из z_e (до квантования)
        x_hat_q: реконструкция из z_q (после квантования)
        z_e: латентные представления до квантования
        z_q: латентные представления после квантования
        indices: индексы победителей на SOM сетке
        logits: предсказания классификатора
        targets: истинные метки классов
        som_layer: SOM слой для расчета соседей
        alpha: вес commitment loss
        beta: вес SOM loss
        gamma: вес classification loss
    
    Returns:
        total_loss, l_reconstruction, l_commitment, l_som, l_cls
    """
    # 1. Reconstruction Loss (MSE между оригиналом и обеими реконструкциями)
    l_rec_e = F.mse_loss(x_hat_e, x)
    l_rec_q = F.mse_loss(x_hat_q, x)
    l_reconstruction = l_rec_e + l_rec_q
    
    # 2. Commitment Loss (регуляризация квантования)
    l_commitment = F.mse_loss(z_e, z_q)
    
    # 3. SOM Loss (регуляризация соседей)
    neighbor_indices = som_layer.get_neighbors(indices)
    neighbors = som_layer.embeddings[neighbor_indices]
    z_e_target = z_e.detach().view(z_e.shape[0], -1).unsqueeze(1)
    l_som = torch.mean((neighbors - z_e_target)**2)
    
    # 4. Classification Loss (кросс-энтропия)
    l_cls = F.cross_entropy(logits, targets)
    
    # Итоговый лосс с учетом всех компонентов
    total_loss = l_reconstruction + alpha * l_commitment + beta * l_som + gamma * l_cls
    
    return total_loss, l_reconstruction, l_commitment, l_som, l_cls

## Training Functions

### Map init func

In [ ]:
def restart_som_with_data(model, train_loader, device):
    """
    Инициализация весов SOM реальными данными из первого батча
    """
    model.eval()
    with torch.no_grad():
        # Берем один батч
        images, _, _ = next(iter(train_loader))
        z_e = model.encoder(images.to(device)) # [B, C, H, W]
        z_e_flat = z_e.view(z_e.size(0), -1)   # [B, Dim]
        
        num_embeddings = model.som.embeddings.size(0)
        
        # Если батч меньше, чем число узлов, просто повторяем его
        indices = torch.arange(num_embeddings) % z_e_flat.size(0)
        initial_weights = z_e_flat[indices]
        
        # Прямое копирование в веса
        model.som.embeddings.data.copy_(initial_weights)
    print(f"Карта SOM инициализирована {num_embeddings} векторами из данных.")

### Default train func

In [ ]:
def train_som_vae(model, train_dataset, val_dataset, optimizer, device, 
                  epochs=50, batch_size=128, alpha=1.0, beta=1.0, gamma=1.0):
    """
    Обучение SOM-VAE модели
    """
    train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size)
    
    # История обучения для визуализации
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'nmi': []
    }
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = {'total': 0, 'rec': 0, 'comm': 0, 'som': 0, 'cls': 0}
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        
        for data, labels, _ in pbar:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # Прямой проход
            x_hat_e, x_hat_q, z_e, z_q, indices, logits = model(data)
            
            # Расчет лосса
            loss, l_rec, l_comm, l_som, l_cls = som_vae_loss(
                data, x_hat_e, x_hat_q, z_e, z_q, indices, logits, labels, model.som,
                alpha=alpha, beta=beta, gamma=gamma
            )
            
            loss.backward()
            optimizer.step()
            
            # Считаем точность
            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
            # Накопление лоссов
            train_losses['total'] += loss.item()
            train_losses['rec'] += l_rec.item()
            train_losses['comm'] += l_comm.item()
            train_losses['som'] += l_som.item()
            train_losses['cls'] += l_cls.item()
            
            pbar.set_postfix({
                'L': f"{loss.item():.3f}",
                'Rec': f"{l_rec.item():.3f}",
                'Comm': f"{l_comm.item():.3f}",
                'som': f"{l_som.item():.3f}",
                'Cls': f"{l_cls.item():.3f}",
                'Acc': f"{100 * train_correct / train_total:.1f}%"
            })

        # --- ФАЗА ВАЛИДАЦИИ ---
        model.eval()
        val_correct = 0
        val_total = 0
        val_total_loss = 0
        all_indices = []
        all_labels = []
        
        with torch.no_grad():
            for data, labels, _ in tqdm(val_loader, desc='Validation', leave=False):
                data, labels = data.to(device), labels.to(device)
                x_hat_e, x_hat_q, z_e, z_q, indices, logits = model(data)
                
                # Расчет лосса для валидации
                loss, _, _, _, _ = som_vae_loss(
                    data, x_hat_e, x_hat_q, z_e, z_q, indices, logits, labels, model.som,
                    alpha=alpha, beta=beta, gamma=gamma
                )
                
                # Точность на валидации
                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                val_total_loss += loss.item()

                all_indices.append(indices.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
                
        # Расчет метрик
        flat_indices = np.concatenate(all_indices).ravel()
        flat_labels = np.concatenate(all_labels).ravel()
        current_nmi = normalized_mutual_info_score(flat_labels, flat_indices)

        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total
        
        # Сохранение истории
        history['train_loss'].append(train_losses['total'] / len(train_loader))
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_total_loss / len(val_loader))
        history['val_acc'].append(val_acc)
        history['nmi'].append(current_nmi)
        
        tqdm.write(f"Summary Epoch {epoch}:")
        tqdm.write(f"Train Loss: {history['train_loss'][-1]:.4f}| Val Loss: {history['val_loss'][-1]:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | NMI: {current_nmi:.4f}")

        if epoch % 5 == 1:
            for cls_num in range(10):
                visualize_som_sample(model, train_dataset, cls_num)
                visualize_som_sample(model, val_dataset, cls_num)
            plot_som_map(model)
        
    return history

### Train func for pretrained model

In [ ]:
def train_som_vae_pretrained(model, train_dataset, val_dataset, optimizer, device, 
                  epochs=10, batch_size=128, info_interval=5, alpha=1.0, beta=1.0, gamma=1.0):
    """
    Обучение SOM-VAE модели
    """
    train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size)
    
    # История обучения для визуализации
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'nmi': []
    }
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = {'total': 0, 'rec': 0, 'comm': 0, 'som': 0, 'cls': 0}
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        
        for data, labels, _ in pbar:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # Прямой проход
            x_hat_e, x_hat_q, z_e, z_q, indices, logits = model(data)
            
            # Расчет лосса
            loss, l_rec, l_comm, l_som, l_cls = som_vae_loss(
                data, x_hat_e, x_hat_q, z_e, z_q, indices, logits, labels, model.som,
                alpha=alpha, beta=beta, gamma=gamma
            )
            
            loss.backward()
            optimizer.step()
            
            # Считаем точность
            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
            # Накопление лоссов
            train_losses['total'] += loss.item()
            train_losses['rec'] += l_rec.item()
            train_losses['comm'] += l_comm.item()
            train_losses['som'] += l_som.item()
            train_losses['cls'] += l_cls.item()
            
            pbar.set_postfix({
                'L': f"{loss.item():.3f}",
                'Rec': f"{l_rec.item():.3f}",
                'Comm': f"{l_comm.item():.3f}",
                'som': f"{l_som.item():.3f}",
                'Cls': f"{l_cls.item():.3f}",
                'Acc': f"{100 * train_correct / train_total:.1f}%"
            })

        # --- ФАЗА ВАЛИДАЦИИ ---
        model.eval()
        val_correct = 0
        val_total = 0
        val_total_loss = 0
        all_indices = []
        all_labels = []
        
        with torch.no_grad():
            for data, labels, _ in tqdm(val_loader, desc='Validation', leave=False):
                data, labels = data.to(device), labels.to(device)
                x_hat_e, x_hat_q, z_e, z_q, indices, logits = model(data)
                
                # Расчет лосса для валидации
                loss, _, _, _, _ = som_vae_loss(
                    data, x_hat_e, x_hat_q, z_e, z_q, indices, logits, labels, model.som,
                    alpha=alpha, beta=beta, gamma=gamma
                )
                
                # Точность на валидации
                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                val_total_loss += loss.item()

                all_indices.append(indices.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
                
        # Расчет метрик
        flat_indices = np.concatenate(all_indices).ravel()
        flat_labels = np.concatenate(all_labels).ravel()
        current_nmi = normalized_mutual_info_score(flat_labels, flat_indices)

        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total
        
        # Сохранение истории
        history['train_loss'].append(train_losses['total'] / len(train_loader))
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_total_loss / len(val_loader))
        history['val_acc'].append(val_acc)
        history['nmi'].append(current_nmi)
        
        tqdm.write(f"Summary Epoch {epoch}:")
        tqdm.write(f"Train Loss: {history['train_loss'][-1]:.4f}| Val Loss: {history['val_loss'][-1]:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | NMI: {current_nmi:.4f}")

        if epoch % info_interval == 0:
            plot_som_reconstruction_map(model, figsize=(20, 20))
        
    return history

In [ ]:
def visualize_som_sample(model, dataset, class_num=None, device="cuda"):
    model.eval()

    # --- 1. Случайный пример ---
    if class_num is not None:
        stop_fl = False
        try_cnt = 0
        while not stop_fl:
            try_cnt += 1
            idx = random.randint(0, len(dataset) - 1)
            if dataset[idx][1] == class_num:
                stop_fl = True
            elif try_cnt > 1000:
                raise Exception('too much attempts')
    else:
        idx = random.randint(0, len(dataset) - 1)

    sample = dataset[idx]
    if isinstance(sample, tuple):
        x = sample[0]
        label = sample[1] if len(sample) > 1 else None
    else:
        x = sample
        label = None

    x = x.unsqueeze(0).to(device)

    # --- 2. Прогон через модель ---
    with torch.no_grad():
        x_hat_e, x_hat_q, z_e, z_q, indices, logits = model(x)

    # --- 3. Активации SOM ---
    activations = model.som.calc_activations(z_e)  # [1, K]
    grid_h, grid_w = model.som.grid_h, model.som.grid_w
    act_map = activations[0].view(grid_h, grid_w).detach().cpu()

    winner = indices[0].item()
    win_r = winner // grid_w
    win_c = winner % grid_w

    # --- 4. Победивший embedding ---
    emb = model.som.embeddings[winner]  # [C*H*W]
    emb = emb.view(1, model.som.c, model.som.h, model.som.w)

    with torch.no_grad():
        winner_recon = model.decode(emb.to(device)).cpu()

    # --- 5. Денормализация ---
    def denorm(img):
        img = img.squeeze().permute(1, 2, 0)
        img = img * 0.5 + 0.5
        return img.clamp(0, 1)

    x = denorm(x.cpu())
    x_hat_e = denorm(x_hat_e.cpu())
    x_hat_q = denorm(x_hat_q.cpu())
    winner_recon = denorm(winner_recon)

    # --- 6. Визуализация ---
    fig, axs = plt.subplots(2, 4, figsize=(20, 10))
    
    axs[0][0].imshow(x)
    axs[0][0].axis('off')
    axs[0][0].set_title(r"Исходная картинка")

    axs[0][1].imshow(x_hat_e)
    axs[0][1].axis('off')
    axs[0][1].set_title(r"Реконструкция сразу после кодировщика($\hat{x_e}$)")

    axs[0][2].imshow(x_hat_q)
    axs[0][2].axis('off')
    axs[0][2].set_title(r"Реконструкция узла-победителя($\hat{x_q}$)"+ f", индекс{indices[0]}")

    act_map = axs[0][3].imshow(act_map, cmap='magma')
    fig.colorbar(act_map, ax=axs[0][3])
    axs[0][3].set_title(r"Карта активации узлов")

    neighbors = model.som.get_neighbors(indices)[0]

    for num, neighbor in enumerate(neighbors):
        emb = model.som.embeddings[neighbor]
        emb = emb.view(1, model.som.c, model.som.h, model.som.w)
        
        with torch.no_grad():
            neighbor_recon = model.decode(emb.to(device)).cpu()

        x_hat_neighbor = denorm(neighbor_recon)

        axs[1][num].imshow(x_hat_neighbor)
        axs[1][num].axis('off')
        axs[1][num].set_title(f"Реконструкция  соседа по индексу {neighbor}")

    fig.suptitle(f"Отчет по реконструкции для класса {dataset[idx][-1]}")
    fig.tight_layout()
    plt.show()

In [ ]:
def plot_som_map(model, device='cuda'):
    som_map = model.som.embeddings.view(-1, model.som.c, model.som.h, model.som.w)
    with torch.no_grad():
        som_cls = model.classifier(som_map.to(device)).cpu()

    fig, ax = plt.subplots(3, 4, figsize=(20, 15))

    for cls_num in range(10):
        som_cls_act = som_cls[:,cls_num]
        temperature = ax[cls_num // 4][cls_num % 4].imshow(som_cls_act.reshape(model.som.grid_h, model.som.grid_w))
        fig.colorbar(temperature, ax=ax[cls_num // 4][cls_num % 4])
        ax[cls_num // 4][cls_num % 4].set_title(f"Карта логитов класса {cls_num} для som-map")

    ax[2][2].imshow(som_cls.argmax(axis=1).reshape(model.som.grid_h, model.som.grid_w), cmap='rainbow')
    ax[2][3].imshow(som_cls.argmin(axis=1).reshape(model.som.grid_h, model.som.grid_w), cmap='rainbow')
    fig.tight_layout()
    plt.show()

In [ ]:
@torch.no_grad()
def plot_som_reconstruction_map(model, figsize=(12, 12), save_path=None):
    """
    Визуализирует карту реконструкций: каждый узел SOM → декодированное изображение.
    Результат: сетка [grid_h x grid_w] изображений.
    """
    model.eval()
    device = next(model.parameters()).device
    
    # 1. Берём все веса SOM и приводим к форме [N, C, H, W]
    embeddings = model.som.embeddings  # [N, C*H*W]
    z_q = embeddings.view(-1, model.som.c, model.som.h, model.som.w).to(device)
    
    # 2. Декодируем все шаблоны сразу
    reconstructions = model.decode(z_q)  # [N, 3, 64, 64]
    
    # 3. Нормализуем из [-1, 1] (Tanh) → [0, 1] для визуализации
    reconstructions = (reconstructions + 1) / 2.0
    
    # 4. Собираем мозаику
    grid_h, grid_w = model.som.grid_h, model.som.grid_w
    img_h, img_w = reconstructions.shape[2], reconstructions.shape[3]
    
    mosaic = torch.zeros(3, grid_h * img_h, grid_w * img_w)
    for idx in range(grid_h * grid_w):
        r = idx // grid_w
        c = idx % grid_w
        mosaic[:, r*img_h:(r+1)*img_h, c*img_w:(c+1)*img_w] = reconstructions[idx]
    
    # 5. Отображаем
    plt.figure(figsize=figsize)
    plt.imshow(mosaic.permute(1, 2, 0).cpu())
    plt.axis('off')
    plt.title(f'SOM Reconstruction Map ({grid_h}×{grid_w} nodes)')
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()

## Save/Load functions

In [ ]:
def save_model(model, path, metadata=None):
    """Сохраняет модель и опциональные метаданные"""
    torch.save({
        'model_state_dict': model.state_dict(),
        'metadata': metadata or {}
    }, path)

def load_model(path, device='cpu'):
    """Загружает модель на указанный девайс"""
    
    checkpoint = torch.load(path, map_location=device)
    model = EuroSAT_GlobalSOM_Deep()  # передай свои параметры при необходимости
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model, checkpoint['metadata']

## Model Initialization and Training

### Подбор параметров через optuna

In [ ]:
grid_size = (16, 32)           # Размер SOM сетки
latent_dim = (64, 8, 8)     # (channels, height, width) латентного представления
num_classes = 10             # Количество классов в EuroSAT


batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def validate(model, val_loader, alpha, beta, gamma=0):
    model.eval()
    val_correct = 0
    val_total = 0
    val_total_loss = 0
    all_indices = []
    all_labels = []
    
    with torch.no_grad():
        for data, labels, _ in val_loader:
            data, labels = data.to(device), labels.to(device)
            x_hat_e, x_hat_q, z_e, z_q, indices, logits = model(data)
            
            # Расчет лосса для валидации
            loss, l_reconstruction, _, _, _ = som_vae_loss(
                data, x_hat_e, x_hat_q, z_e, z_q, indices, logits, labels, model.som,
                alpha=alpha, beta=beta, gamma=gamma
            )
            
            # Точность на валидации
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            val_total_loss += l_reconstruction.item()

            all_indices.append(indices.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    # Расчет метрик
    flat_indices = np.concatenate(all_indices).ravel()
    flat_labels = np.concatenate(all_labels).ravel()
    current_nmi = normalized_mutual_info_score(flat_labels, flat_indices)

    val_acc = 100 * val_correct / val_total

    return val_total_loss / len(val_loader) + (1 - current_nmi)

def objective(trial):
    # Гиперпараметры, которые реально влияют на качество:
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    alpha = trial.suggest_float("alpha", 0.01, 1e+1, log=True)   # commitment loss weight
    beta = trial.suggest_float("beta", 0.01, 1e+1, log=True)     # SOM loss weight
    
    
    # Инициализация модели
    model = EuroSAT_GlobalSOM_Deep(
        in_channels=3, 
        grid_size=grid_size, 
        latent_dim=latent_dim, 
        num_classes=num_classes
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Короткое обучение (1-2 эпохи для быстрой оценки)
    for epoch in range(4):
        for batch in train_loader:
            x = batch[0].to(device)
            labels = batch[1].to(device)
            x_hat_e, x_hat_q, z_e, z_q, indices, logits = model(x)
            
            # Взвешиваем компоненты лосса как в статье:
            loss, l_rec, l_comm, l_som, l_cls = som_vae_loss(
                x, x_hat_e, x_hat_q, z_e, z_q, indices, logits, labels, model.som,
                alpha=alpha, beta=beta, gamma=0
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Валидация: средний реконструкционный лосс + кластеризация (NMI если есть лейблы)
    val_loss = validate(model, val_loader, alpha=alpha, beta=beta)
    
    return val_loss  # Optuna минимизирует

# Запуск
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Лучшие параметры:", study.best_params)

### Первая тренировка(учим реконструкцию)

In [ ]:
# Гиперпараметры модели
grid_size = (16, 16)           # Размер SOM сетки
latent_dim = (32, 4, 4)     # (channels, height, width) латентного представления
num_classes = 10             # Количество классов в EuroSAT

# Параметры обучения
batch_size = 128
epochs = 100
learning_rate = 5e-4

# Веса компонентов лосса
alpha = 2e-2  # Commitment loss weight
beta = 2e-2   # SOM loss weight
gamma = 0  # Classification loss weight

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Создание модели
model = EuroSAT_GlobalSOM_Deep(
    in_channels=3, 
    grid_size=grid_size, 
    latent_dim=latent_dim, 
    num_classes=num_classes
).to(device)


# Создание оптимизатора
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"SOM grid size: {grid_size[0]}x{grid_size[1]} = {grid_size[0] * grid_size[1]} nodes")
print(f"Latent dimension: {latent_dim}")

In [ ]:
# Инициализация SOM весами из данных
train_loader_init = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
restart_som_with_data(model, train_loader_init, device)



print("Модель готова к обучению!")

In [ ]:
# Запуск обучения
history = train_som_vae(
    model=model, 
    train_dataset=train_dataset, 
    val_dataset=val_dataset, 
    optimizer=optimizer, 
    device=device, 
    epochs=epochs, 
    batch_size=batch_size, 
    alpha=alpha, 
    beta=beta, 
    gamma=gamma
)

In [ ]:
# Сохранение
save_model(model, 'checkpoints/som_vae_no_cls_16x16.pth', metadata={'epoch': 100, 'grid_size': (16, 16), 'latent_dim': (32, 4, 4)})

# Загрузка для инференса
model, meta = load_model('checkpoints/som_vae_no_cls_16x16.pth', device='cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
plot_som_reconstruction_map(model, figsize=(20, 20), save_path='som_map_no_cls_16x16.png')

#### Дообучение

In [ ]:
model, meta = load_model('checkpoints/som_vae_no_cls_16x16_finetune_new.pth', device='cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Параметры обучения
batch_size = 128
epochs = 5
learning_rate = 1e-2
info_interval=1

# Веса компонентов лосса
alpha = 1e-0  # Commitment loss weight
beta = 1e-0   # SOM loss weight
gamma = 0  # Classification loss weight

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Запуск обучения
history_pretrained = train_som_vae_pretrained(
    model=model, 
    train_dataset=train_dataset, 
    val_dataset=val_dataset, 
    optimizer=optimizer, 
    device=device, 
    epochs=epochs,
    batch_size=batch_size,
    info_interval=info_interval,
    alpha=alpha, 
    beta=beta, 
    gamma=gamma
)

In [ ]:
# Сохранение
save_model(model, 'checkpoints/som_vae_no_cls_16x16_finetune_new_high_lr.pth', metadata={'epoch': 5, 'grid_size': (16, 16), 'latent_dim': (32, 4, 4)})

# Загрузка для инференса
model, meta = load_model('checkpoints/som_vae_no_cls_16x16_finetune_new_high_lr.pth', device='cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
plot_som_reconstruction_map(model, figsize=(20, 20), save_path='som_map_no_cls_16x16_finetune_new_high_lr.png')

### Вторая тренировка(учим другую модель на лоссе с классификацией)

In [ ]:
# Гиперпараметры модели
grid_size = (32, 32)           # Размер SOM сетки
latent_dim = (32, 4, 4)     # (channels, height, width) латентного представления
num_classes = 10             # Количество классов в EuroSAT

# Параметры обучения
batch_size = 128
epochs = 50
learning_rate = 5e-4

# Веса компонентов лосса
alpha = 1e-2  # Commitment loss weight
beta = 1e-2   # SOM loss weight
gamma = 1e-3  # Classification loss weight

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Создание модели
model = EuroSAT_GlobalSOM_Deep(
    in_channels=3, 
    grid_size=grid_size, 
    latent_dim=latent_dim, 
    num_classes=num_classes
).to(device)


# Создание оптимизатора
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"SOM grid size: {grid_size[0]}x{grid_size[1]} = {grid_size[0] * grid_size[1]} nodes")
print(f"Latent dimension: {latent_dim}")
# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Создание оптимизатора
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"SOM grid size: {grid_size[0]}x{grid_size[1]} = {grid_size[0] * grid_size[1]} nodes")
print(f"Latent dimension: {latent_dim}")

In [ ]:
# Инициализация SOM весами из данных
train_loader_init = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
restart_som_with_data(model, train_loader_init, device)

print("Модель готова к обучению!")

In [ ]:
# Запуск обучения
history_map_train = train_som_vae(
    model=model, 
    train_dataset=train_dataset, 
    val_dataset=val_dataset, 
    optimizer=optimizer, 
    device=device, 
    epochs=epochs, 
    batch_size=batch_size, 
    alpha=alpha, 
    beta=beta, 
    gamma=gamma
)

In [ ]:
# Сохранение
save_model(model, 'checkpoints/som_vae_cls_32x32.pth', metadata={'epoch': 50, 'grid_size': (32, 32), 'latent_dim': (32, 4, 4)})

# Загрузка для инференса
model, meta = load_model('checkpoints/som_vae_cls_32x32.pth', device='cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
plot_som_reconstruction_map(model, figsize=(20, 20), save_path='som_map_32x32_cls.png')

## Training Results Visualization

In [ ]:
# Визуализация истории обучения
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Accuracy
axes[0, 1].plot(history['train_acc'], label='Train Acc')
axes[0, 1].plot(history['val_acc'], label='Val Acc')
axes[0, 1].set_title('Training and Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# NMI
axes[1, 0].plot(history['nmi'], label='NMI', color='purple')
axes[1, 0].set_title('Normalized Mutual Information')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('NMI Score')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Combined metrics
axes[1, 1].plot(history['val_acc'], label='Val Acc', color='blue')
axes[1, 1].plot(history['nmi'], label='NMI', color='purple')
axes[1, 1].set_title('Validation Metrics')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print(f"Final Results:")
print(f"Train Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"Validation Accuracy: {history['val_acc'][-1]:.2f}%")
print(f"Final NMI: {history['nmi'][-1]:.4f}")

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

def analyze_latent_similarity(z_e, z_q, show=True):
    """
    z_e, z_q: [1, C, H, W]
    """
    z_e = z_e.detach().cpu()
    z_q = z_q.detach().cpu()

    # --- flatten ---
    ze_flat = z_e.view(1, -1)
    zq_flat = z_q.view(1, -1)

    # --- Метрики ---
    mse = F.mse_loss(z_q, z_e).item()
    l2 = torch.norm(ze_flat - zq_flat, dim=1).item()
    cos = F.cosine_similarity(ze_flat, zq_flat).item()

    # --- Spatial error map ---
    spatial_err = ((z_e - z_q) ** 2).mean(dim=1)[0]  # [H, W]

    # --- Channel-wise error ---
    channel_err = ((z_e - z_q) ** 2).mean(dim=(2, 3))[0]  # [C]

    if show:
        fig, axs = plt.subplots(1, 3, figsize=(14, 4))

        # spatial
        im0 = axs[0].imshow(spatial_err, cmap="inferno")
        axs[0].set_title("Spatial MSE (8×8)")
        plt.colorbar(im0, ax=axs[0])

        # channels
        axs[1].plot(channel_err.numpy())
        axs[1].set_title("Channel-wise MSE")
        axs[1].set_xlabel("Channel")
        axs[1].set_ylabel("Error")

        # summary text
        axs[2].axis("off")
        axs[2].text(0.05, 0.7, f"MSE: {mse:.6f}", fontsize=14)
        axs[2].text(0.05, 0.5, f"L2 norm: {l2:.4f}", fontsize=14)
        axs[2].text(0.05, 0.3, f"Cos sim: {cos:.4f}", fontsize=14)

        plt.tight_layout()
        plt.show()

    return {
        "mse": mse,
        "l2": l2,
        "cosine": cos,
        "spatial_err": spatial_err,
        "channel_err": channel_err
    }

def analyze_random(model, dataset, device="cuda"):
    model.eval()

    # --- 1. Случайный пример ---
    idx = random.randint(0, len(dataset) - 1)

    sample = dataset[idx]
    if isinstance(sample, tuple):
        x = sample[0]
        label = sample[1] if len(sample) > 1 else None
    else:
        x = sample
        label = None

    x = x.unsqueeze(0).to(device)

    # --- 2. Прогон через модель ---
    with torch.no_grad():
        x_hat_e, x_hat_q, z_e, z_q, indices, logits = model(x)

    return analyze_latent_similarity(z_e, z_q)

In [ ]:
analyze_random(model, val_dataset)

In [ ]:
visualize_som_sample(model, train_dataset)

# Создание Diffusion Model

In [ ]:
# Diffusion model imports
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('')), '../src'))
from diffusion_vae import create_diffusion_vae, train_diffusion_vae

In [ ]:
# Создание conditional diffusion model
diffusion_vae = create_diffusion_vae(model, device='cuda', timesteps=100)
print("Diffusion model created successfully!")

In [ ]:
# Создание и обучение
diffusion_vae, diffusion_history = train_diffusion_vae(
    som_vae_model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    epochs=2,
    batch_size=32,
    lr=2e-4,        # Снизили с 1e-3 для стабильности
    device='cuda',
    timesteps=1000  # Увеличили до стандарта (косинус это позволяет)
)

In [ ]:
def visualize_enhancement(original, som_recon, enhanced):
    num_samples = len(original)
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    
    for i in range(num_samples):
        imgs = [original[i], som_recon[i], enhanced[i]]
        titles = ['Original', 'SOM-VAE', 'Enhanced']
        for j in range(3):
            # Денормализация из [-1, 1] в [0, 1] для корректного отображения
            img = imgs[j].permute(1, 2, 0).cpu().numpy()
            img = np.clip(img * 0.5 + 0.5, 0, 1)
            
            axes[i, j].imshow(img)
            axes[i, j].set_title(titles[j])
            axes[i, j].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================
# INFERENCE CELL (JUPYTER)
# ==============================

# Переводим модели в eval
model.eval()
diffusion_vae.model.eval()

device = 'cuda'

# Сколько примеров показать
num_samples = 10

# Выбираем случайные индексы
indices = np.random.choice(len(val_dataset), num_samples, replace=False)

original_images = []
som_reconstructions = []
enhanced_images = []

# ... ваш код выше ...
with torch.no_grad():
    for idx in indices:
        x, label, name = val_dataset[idx]
        x = x.unsqueeze(0).to(device)

        # ====== SOM-VAE ======
        x_hat_e, _, _, _, _, _ = model(x)
        x_hat = x_hat_e 

        # ====== DIFFUSION ======
        # Оставляем пустые timesteps, чтобы использовать те же 1000 шагов
        # Либо пишем timesteps=100, если нужно ОЧЕНЬ быстро, но качество упадет
        r0, enhanced = diffusion_vae.sample(x_hat) 

        original_images.append(x.cpu().squeeze(0))
        som_reconstructions.append(x_hat.cpu().squeeze(0))
        enhanced_images.append(enhanced.cpu().squeeze(0))

# Визуализация
visualize_enhancement(
    original_images,
    som_reconstructions,
    enhanced_images
)


# Сохранение моделей

## SOM-VAE

In [ ]:
import os
import torch
import json
from datetime import datetime

def save_som_vae(model, optimizer=None, path='checkpoints/', prefix='som_vae'):
    """
    Сохраняет полный чекпоинт модели с контекстом для генерации
    """
    os.makedirs(path, exist_ok=True)
    
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'arch': {
            'grid_h': model.som.grid_h,
            'grid_w': model.som.grid_w,
            'latent_dim': (model.som.c, model.som.h, model.som.w),
            'in_channels': model.encoder[0].in_channels,
            'num_classes': model.classifier[-1].out_features,
        },
        'training': {
            'alpha': getattr(model, 'alpha', None),
            'beta': getattr(model, 'beta', None),
            'gamma': getattr(model, 'gamma', None),
        },
        'timestamp': datetime.now().isoformat(),
    }
    
    # Если есть оптимизатор — сохраняем для продолжения обучения
    if optimizer is not None:
        checkpoint['optimizer_state_dict'] = optimizer.state_dict()
    
    # Формируем имя файла: som_vae_epoch_15_20240615_1430.pt
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    filename = f"{prefix}_{timestamp}.pt"
    filepath = os.path.join(path, filename)
    
    torch.save(checkpoint, filepath)
    print(f"✅ Модель сохранена: {filepath}")
    
    # Сохраняем архитектуру в отдельный JSON для быстрого просмотра
    with open(filepath.replace('.pt', '_arch.json'), 'w') as f:
        json.dump(checkpoint['arch'], f, indent=2)
    
    return filepath


def load_som_vae(filepath, device='cpu', inference_only=False):
    """
    Загружает модель с восстановлением архитектуры
    
    Args:
        inference_only: если True — не загружает оптимизатор (быстрее для демо)
    
    Returns:
        model, optimizer (или None), checkpoint
    """
    checkpoint = torch.load(filepath, map_location=device)
    
    # Восстанавливаем архитектуру
    model = EuroSAT_GlobalSOM_Deep(
        in_channels=checkpoint['arch']['in_channels'],
        grid_size=(checkpoint['arch']['grid_h'], checkpoint['arch']['grid_w']),
        latent_dim=checkpoint['arch']['latent_dim'],
        num_classes=checkpoint['arch']['num_classes']
    ).to(device)
    
    # Загружаем веса
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Для инференса — сразу в eval mode
    if inference_only:
        model.eval()
        return model, None, checkpoint
    
    # Для продолжения обучения — загружаем оптимизатор
    optimizer = torch.optim.Adam(model.parameters(), lr=checkpoint['training'].get('learning_rate', 1e-4))
    if 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    return model, optimizer, checkpoint

In [ ]:
save_som_vae(model=model, path='checkpoints/', prefix='som_vae_1st_save')

## Diffuse VAE

In [ ]:
import torch
from datetime import datetime
import os

def save_diffusion_model(diffusion_vae, path='diffusion_model.pt'):
    """
    Сохраняет только веса диффузионной модели + буферы шедулера
    """
    checkpoint = {
        'unet_state_dict': diffusion_vae.model.state_dict(),
        'scheduler_buffers': {
            'betas': diffusion_vae.scheduler.betas.cpu(),
            'alphas': diffusion_vae.scheduler.alphas.cpu(),
            'alphas_cumprod': diffusion_vae.scheduler.alphas_cumprod.cpu(),
            'sqrt_one_minus_alphas_cumprod': diffusion_vae.scheduler.sqrt_one_minus_alphas_cumprod.cpu(),
        },
        'timesteps': diffusion_vae.scheduler.timesteps,
        'beta_schedule': 'cosine',  # хардкод из твоего кода
        'timestamp': datetime.now().isoformat(),
    }
    torch.save(checkpoint, path)
    print(f"✅ Diffusion model saved to {path}")
    return path


def load_diffusion_model(path, device='cpu'):
    """
    Загружает диффузионную модель в инференс-режиме
    """
    ckpt = torch.load(path, map_location=device)
    
    # Создаём модель с теми же параметрами
    model = LightConditionalUNet(time_dim=256).to(device)
    model.load_state_dict(ckpt['unet_state_dict'])
    model.eval()
    
    # Создаём шедулер и копируем буферы
    scheduler = DiffusionScheduler(
        timesteps=ckpt['timesteps'],
        beta_schedule=ckpt['beta_schedule']
    ).to(device)
    
    # Восстанавливаем точные буферы (важно для воспроизводимости!)
    for name, buf in ckpt['scheduler_buffers'].items():
        getattr(scheduler, name).data.copy_(buf.to(device))
    
    # Оборачиваем
    diffusion_vae = ConditionalDiffusionVAE(model, scheduler, device)
    
    return diffusion_vae

In [ ]:
# Сохранение после обучения
save_diffusion_model(diffusion_vae, 'checkpoints/diffusion_eurosat.pt')

# # Загрузка для генерации
# diffusion_vae = load_diffusion_model('models/diffusion_eurosat.pt', device='cuda')

# # Генерация (с твоей уже загруженной SOM-VAE)
# x_hat = som_vae.decode(z_from_som_node)          # [1, 3, 64, 64]
# residual, x_final = diffusion_vae.sample(x_hat)  # уточнённое изображение